# LAB 05 - Bài 19: So sánh RNN, LSTM và GRU

Notebook chuẩn bị cùng dữ liệu train/validation/test, tokenizer và cấu hình, rồi lần lượt huấn luyện ba recurrent architecture. Chỉ thay đổi recurrent layer.

Bảng cuối báo cáo Parameters, training time, Accuracy, Precision, Recall và F1. Chạy các cell theo thứ tự. Lần đầu cần Internet để tải ACL IMDb (~84 MB); ba lần huấn luyện có thể mất vài phút.

In [1]:
# ============================================================
# BÀI 19 - EXPERIMENT 4: CHUẨN BỊ DỮ LIỆU IMDb
# ACL IMDb Reviews của Stanford; nhãn neg=0, pos=1.
# ============================================================

from pathlib import Path
import tarfile
import urllib.request
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

DATA_URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
DATA_DIR = Path("/content/aclImdb")
ARCHIVE_PATH = Path("/content/aclImdb_v1.tar.gz")

# Tải và giải nén dataset nếu chưa có trong runtime.
if not (DATA_DIR / "train" / "pos").exists():
    if not ARCHIVE_PATH.exists():
        print("Đang tải ACL IMDb từ Stanford (~84 MB)...")
        urllib.request.urlretrieve(DATA_URL, ARCHIVE_PATH)
    print("Đang giải nén dataset...")
    with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
        archive.extractall(path="/content")

def read_split(split):
    texts, labels = [], []
    for label_name, label_id in (("neg", 0), ("pos", 1)):
        folder = DATA_DIR / split / label_name
        for file_path in sorted(folder.glob("*.txt")):
            texts.append(file_path.read_text(encoding="utf-8", errors="replace").strip())
            labels.append(label_id)
    return texts, np.asarray(labels, dtype=np.int32)

all_train_texts, all_train_labels = read_split("train")
test_texts, y_test = read_split("test")

# Tạo cùng một train/validation split để dùng cho cả ba kiến trúc.
train_texts, val_texts, y_train, y_val = train_test_split(
    all_train_texts,
    all_train_labels,
    test_size=0.1,
    random_state=42,
    stratify=all_train_labels,
)

MAX_VOCAB_SIZE = 20_000
MAX_LEN = 200
EMBEDDING_DIM = 64
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001
SEED = 42

# Fit vocabulary chỉ trên train, rồi dùng đúng vocabulary cho val/test.
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

X_train = pad_sequences(train_sequences, maxlen=MAX_LEN, padding="post", truncating="post")
X_val = pad_sequences(val_sequences, maxlen=MAX_LEN, padding="post", truncating="post")
X_test = pad_sequences(test_sequences, maxlen=MAX_LEN, padding="post", truncating="post")

print("Vocabulary size:", vocab_size)
print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Label counts, train (neg=0, pos=1):", np.bincount(y_train))


Đang tải ACL IMDb từ Stanford (~84 MB)...
Đang giải nén dataset...


/tmp/ipykernel_764/3390277211.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(path="/content")


Vocabulary size: 20000
Train: (22500, 200) Validation: (2500, 200) Test: (25000, 200)
Label counts, train (neg=0, pos=1): [11250 11250]


In [2]:
# ============================================================
# BÀI 19 - SO SÁNH RNN, LSTM VÀ GRU
# Chỉ thay recurrent layer; các thiết lập còn lại được giữ cố định.
# ============================================================

import time
import pandas as pd
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Factory tạo model. Mọi model đều dùng cùng Embedding, hidden size,
# Dense output, loss, optimizer, learning rate và dữ liệu.
def build_model(model_name):
    recurrent_layers = {
        "RNN": layers.SimpleRNN,
        "LSTM": layers.LSTM,
        "GRU": layers.GRU,
    }
    recurrent_layer = recurrent_layers[model_name]

    model = models.Sequential([
        layers.Input(shape=(MAX_LEN,)),
        layers.Embedding(vocab_size, EMBEDDING_DIM, mask_zero=True),
        recurrent_layer(64),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

model_results = []
trained_models = {}

for model_name in ("RNN", "LSTM", "GRU"):
    print(f"\n{'=' * 20} {model_name} {'=' * 20}")
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(model_name)
    parameter_count = model.count_params()

    start_time = time.perf_counter()
    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=2,
    )
    training_time = time.perf_counter() - start_time

    probabilities = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
    predictions = (probabilities >= 0.5).astype("int32")

    model_results.append({
        "Model": model_name,
        "Parameters": parameter_count,
        "Training time (seconds)": training_time,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
    })
    trained_models[model_name] = model

results_df = pd.DataFrame(model_results)
print("\nBảng so sánh Bài 19 (test set):")
print(results_df.round(4).to_string(index=False))

OUTPUT_CSV = "/content/lab05_bai19_results.csv"
results_df.to_csv(OUTPUT_CSV, index=False)
print("\nĐã lưu kết quả:", OUTPUT_CSV)

# Khi viết phân tích, so sánh cả Parameters và Training time,
# không kết luận chỉ dựa trên Accuracy.



==================== RNN ====================
Epoch 1/3
352/352 - 39s - 110ms/step - accuracy: 0.5916 - loss: 0.6646 - val_accuracy: 0.5936 - val_loss: 0.6629
Epoch 2/3
352/352 - 37s - 104ms/step - accuracy: 0.7584 - loss: 0.5125 - val_accuracy: 0.7856 - val_loss: 0.4868
Epoch 3/3
352/352 - 37s - 105ms/step - accuracy: 0.8198 - loss: 0.4110 - val_accuracy: 0.8000 - val_loss: 0.4699

==================== LSTM ====================
Epoch 1/3
352/352 - 70s - 199ms/step - accuracy: 0.7613 - loss: 0.4883 - val_accuracy: 0.8460 - val_loss: 0.3604
Epoch 2/3
352/352 - 68s - 192ms/step - accuracy: 0.8972 - loss: 0.2756 - val_accuracy: 0.8508 - val_loss: 0.3816
Epoch 3/3
352/352 - 68s - 192ms/step - accuracy: 0.9271 - loss: 0.2038 - val_accuracy: 0.8212 - val_loss: 0.4922

==================== GRU ====================
Epoch 1/3
352/352 - 74s - 210ms/step - accuracy: 0.7673 - loss: 0.4690 - val_accuracy: 0.8328 - val_loss: 0.4061
Epoch 2/3
352/352 - 81s - 231ms/step - accuracy: 0.8950 - loss: 0.2